In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import pickle
import pandas as pd
from scipy.ndimage import maximum_filter
from IPython.display import display, Image
from google.colab import output
import base64, json

In [58]:
IMG_PATH = "building.jpeg"

img_bgr = cv2.imread(IMG_PATH)

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)

print(f"Image shape: {img_bgr.shape}  (H x W x C)")
plt.figure(figsize=(10,7))
plt.imshow(img_rgb)
plt.title("Input Image"); plt.axis("off"); plt.show()

Image shape: (1200, 1600, 3)  (H x W x C)


<IPython.core.display.Javascript object>

In [59]:

with open(IMG_PATH, "rb") as f:
    b64 = base64.b64encode(f.read()).decode()

js = f"""
async function collectClicks() {{
  const container = document.createElement('div');
  container.style = "display:flex;flex-direction:column;align-items:start;";

  const img = new Image();
  img.src = "data:image/jpeg;base64,{b64}";
  await img.decode();

  const canvas = document.createElement('canvas');
  canvas.width  = img.width;
  canvas.height = img.height;
  canvas.style  = "border:2px solid red; margin-bottom:6px; max-width:100%; height:auto;";
  const ctx = canvas.getContext('2d');
  ctx.drawImage(img, 0, 0);

  const instr = document.createElement('div');
  instr.innerText = "Click actual corners of the building (window corners, wall junctions, ledges). Target: 50 clicks. Press Finish when done.";
  instr.style = "margin-bottom:6px; font-weight:700; font-size:14px; color:darkred;";

  const counter = document.createElement('div');
  counter.style = "margin-bottom:6px; font-size:13px;";
  counter.innerText = "Points: 0";

  const btns   = document.createElement('div');
  btns.style   = "margin-bottom:8px;";
  const finish = document.createElement('button');
  const undo   = document.createElement('button');
  const clear  = document.createElement('button');
  finish.innerText = "Finish";   finish.style = "margin-right:8px; padding:6px 12px; font-size:13px;";
  undo.innerText   = "Undo";     undo.style   = "margin-right:8px; padding:6px 12px; font-size:13px;";
  clear.innerText  = "Clear";   clear.style  = "padding:6px 12px; font-size:13px;";
  btns.appendChild(finish); btns.appendChild(undo); btns.appendChild(clear);

  container.appendChild(instr);
  container.appendChild(counter);
  container.appendChild(canvas);
  container.appendChild(btns);
  document.body.appendChild(container);

  let points = [];

  function redraw() {{
    ctx.clearRect(0, 0, canvas.width, canvas.height);
    ctx.drawImage(img, 0, 0);
    points.forEach((p, i) => {{
      ctx.beginPath();
      ctx.arc(p[0], p[1], 6, 0, 2*Math.PI);
      ctx.fillStyle   = "lime";
      ctx.fill();
      ctx.strokeStyle = "black";
      ctx.lineWidth   = 1.5;
      ctx.stroke();
      ctx.fillStyle = "white";
      ctx.font      = "bold 10px sans-serif";
      ctx.fillText(i+1, p[0]+7, p[1]-4);
    }});
    counter.innerText = "Points: " + points.length;
  }}

  canvas.addEventListener('click', (ev) => {{
    const rect = canvas.getBoundingClientRect();
    const x = Math.round((ev.clientX - rect.left)  * (canvas.width  / rect.width));
    const y = Math.round((ev.clientY - rect.top)   * (canvas.height / rect.height));
    points.push([x, y]);
    redraw();
  }});

  undo.onclick  = () => {{ points.pop(); redraw(); }};
  clear.onclick = () => {{ points = []; redraw(); }};

  return new Promise(resolve => {{
    finish.onclick = () => {{ container.remove(); resolve(JSON.stringify(points)); }};
  }});
}}
collectClicks();
"""

res    = output.eval_js(js)
coords = np.array(json.loads(res), dtype=np.int32)
print(f"Collected {len(coords)} ground truth corners.")
assert 40 <= len(coords) <= 70, f"Expected 40-70 corners, got {len(coords)} — re-annotate"
np.save("ground_truth.npy", coords)
print("Saved ground_truth.npy")

Collected 51 ground truth corners.
Saved ground_truth.npy


In [60]:
gt_raw = np.load("ground_truth.npy", allow_pickle=True)
gt     = [tuple(pt) for pt in gt_raw]
print(f"Ground truth corners: {len(gt)}")


vis_gt = img_rgb.copy()
for i, (x, y) in enumerate(gt):
    cv2.circle(vis_gt, (x, y), 8,  (0, 255, 0), 2)
    cv2.circle(vis_gt, (x, y), 2,  (0, 255, 0), -1)
    cv2.putText(vis_gt, str(i+1), (x+6, y-4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,0), 1)

plt.figure(figsize=(12, 8))
plt.imshow(vis_gt)
plt.title(f"Ground Truth Corners — {len(gt)} points (should all be on real corners)")
plt.axis("off"); plt.tight_layout(); plt.show()

Ground truth corners: 51


<IPython.core.display.Javascript object>

In [61]:


dst = cv2.cornerHarris(gray, blockSize=2, ksize=3, k=0.04)

print(f"Harris response range: min={dst.min():.4f}  max={dst.max():.4f}")
print(f"Pixels with response > 0: {int((dst > 0).sum()):,}")


plt.figure(figsize=(10, 7))
plt.imshow(np.clip(dst, 0, dst.max()*0.1), cmap='hot')
plt.colorbar(label='Harris response')
plt.title("Raw Harris Response Map"); plt.axis("off"); plt.show()

Harris response range: min=-65691964.0000  max=195577328.0000
Pixels with response > 0: 863,477


<IPython.core.display.Javascript object>

In [62]:


print("NMS tuning pickink min_distance that gives 300-800 candidates:\n")
for d in [10, 15, 20, 30, 40, 50, 70, 100]:
    lm   = maximum_filter(dst, size=2*d+1)
    mask = (dst == lm) & (dst > 0)
    flag = "  ← good" if 300 <= int(mask.sum()) <= 800 else ""
    print(f"  min_distance = {d:3d}   candidates = {int(mask.sum()):>6,}{flag}")

NMS tuning pickink min_distance that gives 300-800 candidates:

  min_distance =  10   candidates =  3,413
  min_distance =  15   candidates =  1,652
  min_distance =  20   candidates =    967
  min_distance =  30   candidates =    457  ← good
  min_distance =  40   candidates =    277
  min_distance =  50   candidates =    160
  min_distance =  70   candidates =     85
  min_distance = 100   candidates =     43


In [63]:

MIN_DIST = 30

lm       = maximum_filter(dst, size=2*MIN_DIST+1)
nms_mask = (dst == lm) & (dst > 0)

n_candidates = int(nms_mask.sum())
print(f"NMS with min_distance={MIN_DIST} → {n_candidates:,} candidates")
assert 200 <= n_candidates <= 1500, \
    f"Still {n_candidates} candidates — go back to Block 6 and pick a better MIN_DIST"


vis_nms = img_rgb.copy()
ys, xs  = np.where(nms_mask)
for x, y in zip(xs, ys):
    cv2.circle(vis_nms, (int(x), int(y)), 2, (255, 0, 0), -1)

plt.figure(figsize=(12, 8))
plt.imshow(vis_nms)
plt.title(f"All NMS Candidates ({n_candidates}) before thresholding — red dots")
plt.axis("off"); plt.tight_layout(); plt.show()

NMS with min_distance=30 → 457 candidates


<IPython.core.display.Javascript object>

In [65]:
def detect_corners(dst, nms_mask, frac):
    active = nms_mask & (dst > frac * dst.max())
    ys, xs = np.where(active)
    return list(zip(xs.tolist(), ys.tolist()))


def compute_metrics(pred, gt, match_dist=20):
    if len(pred) == 0:
        return 0.0, 0.0, 0, 0, len(gt)

    pred_np = np.array(pred)
    gt_np   = np.array(gt)
    TP, matched = 0, set()

    for p in pred_np:
        dists = np.linalg.norm(gt_np - p, axis=1)
        for idx in np.argsort(dists):
            if idx not in matched and dists[idx] <= match_dist:
                TP += 1
                matched.add(idx)
                break

    FP = len(pred_np) - TP
    FN = len(gt_np)   - TP
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    return precision, recall, TP, FP, FN

In [66]:

thresholds = np.unique(np.concatenate([
    np.linspace(0.0001, 0.02, 10),
    np.linspace(0.02,   0.15, 6),
    np.linspace(0.15,   0.5,  4),
    np.array([0.6, 0.7, 0.8]),
]))

results = []
print(f"{'frac':>8}  {'prec':>6}  {'rec':>6}  {'TP':>4}  {'FP':>5}  {'FN':>4}  {'det':>5}  {'gt':>4}")
print("-" * 60)

for frac in thresholds:
    pts = detect_corners(dst, nms_mask, frac)
    p, r, TP, FP, FN = compute_metrics(pts, gt, match_dist=20)
    results.append((frac, p, r, TP, FP, FN, pts))
    print(f"{frac:8.4f}  {p:6.3f}  {r:6.3f}  {TP:4d}  {FP:5d}  {FN:4d}  {len(pts):5d}  {len(gt):4d}")

    frac    prec     rec    TP     FP    FN    det    gt
------------------------------------------------------------
  0.0001   0.115   0.824    42    322     9    364    51
  0.0023   0.121   0.824    42    306     9    348    51
  0.0045   0.125   0.824    42    294     9    336    51
  0.0067   0.129   0.824    42    284     9    326    51
  0.0089   0.132   0.824    42    275     9    317    51
  0.0112   0.139   0.824    42    261     9    303    51
  0.0134   0.142   0.824    42    254     9    296    51
  0.0156   0.144   0.824    42    250     9    292    51
  0.0178   0.146   0.824    42    246     9    288    51
  0.0200   0.151   0.824    42    236     9    278    51
  0.0460   0.177   0.784    40    186    11    226    51
  0.0720   0.147   0.529    27    157    24    184    51
  0.0980   0.143   0.412    21    126    30    147    51
  0.1240   0.139   0.314    16     99    35    115    51
  0.1500   0.136   0.275    14     89    37    103    51
  0.2667   0.170   0.157   

In [67]:
os.makedirs("overlays", exist_ok=True)

for frac, p, r, TP, FP, FN, pts in results:
    vis = img_rgb.copy()


    for (x, y) in pts:
        cv2.circle(vis, (x, y), 4, (255, 0, 0), -1)


    for (x, y) in gt:
        cv2.circle(vis, (x, y), 10, (0, 255, 0), 2)


    red_patch   = mpatches.Patch(color='red',   label=f'Detected ({len(pts)})')
    green_patch = mpatches.Patch(color='green', label=f'GT ({len(gt)})')

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(vis)
    ax.legend(handles=[red_patch, green_patch], loc='upper right', fontsize=11)
    ax.set_title(
        f"Threshold = {frac:.4f}  |  Precision = {p:.3f}  |  Recall = {r:.3f}\n"
        f"TP = {TP}   FP = {FP}   FN = {FN}",
        fontsize=12
    )
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(f"overlays/overlay_{frac:.4f}.png", dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()

print(f"Saved {len(results)} overlay images → overlays/")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved 21 overlay images → overlays/


In [69]:

rows = [{
    "threshold_frac":  round(frac, 6),
    "precision":       round(p, 4),
    "recall":          round(r, 4),
    "TP": TP, "FP": FP, "FN": FN,
    "detected_count":  len(pts),
    "gt_count":        len(gt)
} for frac, p, r, TP, FP, FN, pts in results]

df = pd.DataFrame(rows)
df.to_csv("harris_summary.csv", index=False)
print("Saved → harris_summary.csv")
display(df)


with open("harris_results.pkl", "wb") as f:
    pickle.dump(results, f)
print("Saved → harris_results.pkl")

Saved → harris_summary.csv


,threshold_frac,precision,recall,TP,FP,FN,detected_count,gt_count
0,0.000100,0.1154,0.8235,42,322,9,364,51
1,0.002311,0.1207,0.8235,42,306,9,348,51
2,0.004522,0.1250,0.8235,42,294,9,336,51
3,0.006733,0.1288,0.8235,42,284,9,326,51
4,0.008944,0.1325,0.8235,42,275,9,317,51
5,0.011156,0.1386,0.8235,42,261,9,303,51
6,0.013367,0.1419,0.8235,42,254,9,296,51
7,0.015578,0.1438,0.8235,42,250,9,292,51
8,0.017789,0.1458,0.8235,42,246,9,288,51
9,0.020000,0.1511,0.8235,42,236,9,278,51


Saved → harris_results.pkl


In [70]:
fracs      = np.array([r[0] for r in results])
precisions = np.array([r[1] for r in results])
recalls    = np.array([r[2] for r in results])


order        = np.argsort(recalls)
recalls_s    = recalls[order]
precisions_s = precisions[order]
fracs_s      = fracs[order]

auc = np.trapezoid(precisions_s, recalls_s)
print(f"AUC (trapezoidal): {auc:.4f}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recalls_s, precisions_s,
        marker='o', linewidth=2.5, color='steelblue',
        label=f"Harris Corner Detector  (AUC = {auc:.4f})")
ax.fill_between(recalls_s, precisions_s, alpha=0.12, color='steelblue')


for i, (rc, pr, fr) in enumerate(zip(recalls_s, precisions_s, fracs_s)):
    ax.annotate(f"{fr:.3f}", (rc, pr),
                textcoords="offset points", xytext=(5, 4),
                fontsize=7, color='dimgray')

ax.set_xlabel("Recall",    fontsize=13)
ax.set_ylabel("Precision", fontsize=13)
ax.set_title(f"Precision-Recall Curve — Harris Corner Detector\nAUC = {auc:.4f}",
             fontsize=13, fontweight='bold')
ax.set_xlim([-0.02, 1.05])
ax.set_ylim([-0.02, 1.05])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig("PR_curve_harris.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → PR_curve_harris.png")

AUC (trapezoidal): 0.1582


<IPython.core.display.Javascript object>

Saved → PR_curve_harris.png
